# ScenarioMIP + overshoot download

Pulls monthly `tas` and `pr` (`Amon`) for **ssp126 / ssp245 / ssp585 / ssp534-over** from the
pangeo GCS CMIP6 catalogue, for the RAMIP model set, into `raw/scenariomip/{model}/` as one zarr
per run (same layout as `raw/ramip/`, so `member_files`-style loaders work unchanged).

- Up to 10 members per (model, experiment, variable), 2015–2100.
- **CanESM5-1 is not on pangeo** — its parent **CanESM5** stands in, stored under its own name
  (`CanESM5/`); any comparison against the RAMIP CanESM5-1 arms must flag the proxy.
- `ssp534-over` (overshoot) exists for only a subset of models; the availability table below
  records what was actually there.

Idempotent: already-downloaded stores are skipped, so the notebook can be resubmitted after a
timeout and it picks up where it left off.

In [1]:
import os, re, glob, warnings
import numpy as np
import pandas as pd
import xarray as xr

import sys
sys.path.insert(0, '/burg-archive/home/mck2199/summer/bcd_me/code')
from funcs_support import get_params
dir_list = get_params()

warnings.filterwarnings('ignore', category=FutureWarning)

SCEN_DIR = dir_list['raw'] + 'scenariomip/'
DL_DIR   = dir_list['aux'] + 'data_layer/'
os.makedirs(SCEN_DIR, exist_ok=True)
os.makedirs(DL_DIR, exist_ok=True)

RAMIP_MODELS = sorted(os.listdir(dir_list['raw'] + 'ramip/'))
# CanESM5-1 absent from pangeo; parent model stands in (flag downstream!)
SOURCES = sorted({{'CanESM5-1': 'CanESM5'}.get(m, m) for m in RAMIP_MODELS})

EXPS   = ['ssp126', 'ssp245', 'ssp585', 'ssp534-over']
VARS   = ['tas', 'pr']
MAXMEM = 10
SPAN   = ('2015', '2100')


def member_sort(m):
    return [int(x) for x in re.findall(r'\d+', m)]


cat = pd.read_csv('https://storage.googleapis.com/cmip6/cmip6-zarr-consolidated-stores.csv')
sub = cat[(cat.table_id == 'Amon') & cat.variable_id.isin(VARS)
          & cat.experiment_id.isin(EXPS) & cat.source_id.isin(SOURCES)]
# one store per (source, exp, member, var): keep newest version
sub = (sub.sort_values('version')
       .drop_duplicates(['source_id', 'experiment_id', 'member_id', 'variable_id'], keep='last'))

avail = (sub.groupby(['source_id', 'experiment_id', 'variable_id'])
         .member_id.nunique().unstack(fill_value=0))
avail.to_csv(DL_DIR + 'scenariomip_availability.csv')
print(avail)

variable_id                pr  tas
source_id   experiment_id         
CESM2       ssp126          3    3
            ssp245          3    3
            ssp585          3    3
CNRM-ESM2-1 ssp126          5    5
            ssp245         10   10
            ssp534-over     5    5
            ssp585          5    5
CanESM5     ssp126         50   50
            ssp245         50   50
            ssp534-over     5    5
            ssp585         50   50
GISS-E2-1-G ssp126         10   10
            ssp245         25   25
            ssp534-over    11   11
            ssp585         10   10
MIROC6      ssp126         50   50
            ssp245         50   50
            ssp534-over     1    1
            ssp585         50   50
MRI-ESM2-0  ssp126          5    5
            ssp245         10   10
            ssp534-over     1    1
            ssp585          6    6
NorESM2-LM  ssp126          1    1
            ssp245         13   13
            ssp585          0    1
UKESM1-0-LL ssp126  

## Download

One zarr per run, named like the RAMIP archive:
`{var}_Amon_{model}_{exp}_{member}_20150116-21001216.zarr`.

In [2]:
todo, done, failed = 0, 0, []
for (src, exp, var), grp in sub.groupby(['source_id', 'experiment_id', 'variable_id']):
    mems = sorted(grp.member_id.unique(), key=member_sort)[:MAXMEM]
    os.makedirs(f'{SCEN_DIR}{src}', exist_ok=True)
    for mem in mems:
        out = f'{SCEN_DIR}{src}/{var}_Amon_{src}_{exp}_{mem}_20150116-21001216.zarr'
        if os.path.exists(out):
            done += 1
            continue
        todo += 1
        zstore = grp[grp.member_id == mem].zstore.iloc[0]
        try:
            ds = xr.open_zarr(zstore, consolidated=True, storage_options={'token': 'anon'})
            da = ds[[var]].sel(time=slice(*SPAN)).load()
            for v in da.variables:   # v2-source encodings (numcodecs Blosc) break the v3 writer
                da[v].encoding = {}
            da.to_zarr(out + '.tmp', mode='w')
            os.rename(out + '.tmp', out)
            print(f'{src} {exp} {var} {mem}: {da.sizes["time"]} months', flush=True)
        except Exception as e:
            failed.append((src, exp, var, mem, str(e)[:120]))
            print(f'FAILED {src} {exp} {var} {mem}: {e}', flush=True)

print(f'\ndownloaded {todo - len(failed)}, cached {done}, failed {len(failed)}')
for f in failed:
    print(' ', f)

CESM2 ssp126 pr r4i1p1f1: 1032 months


/burg-archive/home/mck2199/summer/.venv312/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


CESM2 ssp126 pr r10i1p1f1: 1032 months


/burg-archive/home/mck2199/summer/.venv312/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


CESM2 ssp126 pr r11i1p1f1: 1032 months


...

UKESM1-0-LL ssp585 tas r4i1p1f2: 1032 months


/burg-archive/home/mck2199/summer/.venv312/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


UKESM1-0-LL ssp585 tas r8i1p1f2: 1032 months



downloaded 403, cached 0, failed 4
  ('NorESM2-LM', 'ssp126', 'tas', 'r1i1p1f1', '"Cannot get left slice bound for non-monotonic index with a missing label \'2015\'. Either sort the index or specify an ex')
  ('NorESM2-LM', 'ssp245', 'tas', 'r1i1p1f1', '"Cannot get left slice bound for non-monotonic index with a missing label \'2015\'. Either sort the index or specify an ex')
  ('NorESM2-LM', 'ssp245', 'tas', 'r2i1p1f1', '"Cannot get left slice bound for non-monotonic index with a missing label \'2015\'. Either sort the index or specify an ex')
  ('NorESM2-LM', 'ssp245', 'tas', 'r3i1p1f1', '"Cannot get left slice bound for non-monotonic index with a missing label \'2015\'. Either sort the index or specify an ex')


/burg-archive/home/mck2199/summer/.venv312/lib/python3.12/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [3]:
# inventory of what is on disk now
rows = []
for f in glob.glob(f'{SCEN_DIR}*/[tp]*_Amon_*.zarr'):
    var, _, mod, exp, mem, _ = os.path.basename(f).split('_')
    rows.append({'model': mod, 'exp': exp, 'var': var, 'member': mem})
inv = pd.DataFrame(rows)
inv.to_csv(DL_DIR + 'scenariomip_inventory.csv', index=False)
print(inv.groupby(['model', 'exp', 'var']).member.nunique().unstack(fill_value=0))

var                      pr  tas
model       exp                 
CESM2       ssp126        3    3
            ssp245        3    3
            ssp585        3    3
CNRM-ESM2-1 ssp126        5    5
            ssp245       10   10
            ssp534-over   5    5
            ssp585        5    5
CanESM5     ssp126       10   10
            ssp245       10   10
            ssp534-over   5    5
            ssp585       10   10
GISS-E2-1-G ssp126       10   10
            ssp245       10   10
            ssp534-over  10   10
            ssp585       10   10
MIROC6      ssp126       10   10
            ssp245       10   10
            ssp534-over   1    1
            ssp585       10   10
MRI-ESM2-0  ssp126        5    5
            ssp245       10   10
            ssp534-over   1    1
            ssp585        6    6
NorESM2-LM  ssp126        1    0
            ssp245       10    7
            ssp585        0    1
UKESM1-0-LL ssp126       10   10
            ssp245       10   10
          